# Starplast 0.55.0 clean wheel acceptance


613 release tests and documentation build passed in local packet V2. Its final reporting cell lacked json in the notebook namespace; preserve that failure and complete the receipt here without repeating passed checks. Initial packet retains the two missing-docstring failures, corrected without algorithm changes. Export only tracked package/build inputs; restricted untracked files are not included. The initial builder cell contained an interpreted null separator; corrected to chr(0), with its original failure preserved. The local build/ directory was mistaken for the build distribution; neither interpreter had its CLI. Install the real build tool only in the temporary tools environment; retain V2/V3 failures. Explicitly reuse parent runtime dependencies in the isolated wheel environment. Verify installed resources and new functional cards; no fit or biological admission.

In [1]:
from pathlib import Path
import json,shutil,subprocess,zipfile
prior=ROOT/'results/release_0_55_0_2026_10_08_v2'
assert '613 passed' in (prior/'tests.log').read_text()
assert 'Documentation:' in (prior/'docs.log').read_text()
assert json.loads((prior/'failure.json').read_text())['error']=='NameError'
source=SCRATCH/'source-v4';source.mkdir(exist_ok=False)
files=subprocess.check_output(['git','ls-files','-z','pyproject.toml','README.md','LICENSE','starplast'],cwd=ROOT).decode().split(chr(0))
inputs={}
for name in filter(None,files):
 path=ROOT/name;target=source/name;target.parent.mkdir(parents=True,exist_ok=True)
 shutil.copy2(path,target);inputs[name]=sha(path)
(OUT/'input_manifest.json').write_text(json.dumps(inputs,indent=2)+'\n')
dist=SCRATCH/'dist-v4';toolpython=str(SCRATCH/'tools/bin/python')
run('build-tools',[toolpython,'-m','pip','install','build>=1.2','setuptools>=77','wheel'])
run('build',[toolpython,'-m','build','--no-isolation','--outdir',str(dist)],cwd=source)
run('wheel-check',[python,'scripts/check_wheel.py',str(dist)])
run('metadata',[toolpython,'-m','twine','check','--strict',*[str(p) for p in sorted(dist.iterdir())]])

{'step': 'metadata', 'status': 'passed'}

In [2]:
wheel=next(dist.glob('*.whl'))
required=['starplast/functional_pair_class.py','starplast/group_ratios.py','starplast/data/functional_results.json','starplast/data/functional_agreement.json','starplast/data/functional_agreement_uncertainty.json']
with zipfile.ZipFile(wheel) as archive:
 for name in required:assert archive.read(name)==(ROOT/name).read_bytes(),name
installed=SCRATCH/'installed'
run('installed-env',[python,'-m','venv','--system-site-packages',str(installed)])
import sys,sysconfig
site=installed/'lib'/f'python{sys.version_info.major}.{sys.version_info.minor}'/'site-packages'
(site/'release_dependencies.pth').write_text(sysconfig.get_path('purelib')+'\n')
installedpython=str(installed/'bin/python')
run('install',[installedpython,'-m','pip','install','--no-deps','--force-reinstall',str(wheel)])
code="from pathlib import Path; import starplast; from starplast import paths,functional_results as F,functional_agreement as A,functional_pair_class as C; assert starplast.__version__=='0.55.0'; assert '/installed/lib/' in starplast.__file__; assert paths.check()[0]; b,reason=F.shipped('Pf'); assert not reason and len(b)==2; pair=A.find_pair(b[0],b); r=A.compare(*pair); result,view=C.build(pair,r,'major:2'); assert len(result['rows'])==152; assert len(C.addresses(pair))==19; print(starplast.__version__,starplast.__file__,len(result['rows']))"
run('installed-smoke',[installedpython,'-I','-c',code],cwd=SCRATCH)
run('installed-cli',[str(installed/'bin/starplast-discover'),'--help'],cwd=SCRATCH)
for name,digest in inputs.items():assert sha(ROOT/name)==digest,name
result={'version':'0.55.0','release_checks':613,'documentation':'built','tracked_build_inputs':len(inputs),'wheel_and_sdist':{p.name:{'sha256':sha(p),'size':p.stat().st_size} for p in dist.iterdir()},'installed_wheel':'verified outside source','new_functional_resources':'exact original bytes','broader_goal':'paused','publication':'pending CI and upload'}
(OUT/'acceptance.json').write_text(json.dumps(result,indent=2)+'\n')
result

{'version': '0.55.0', 'release_checks': 613, 'documentation': 'built', 'tracked_build_inputs': 316, 'wheel_and_sdist': {'starplast-0.55.0.tar.gz': {'sha256': 'c0768a6fd5abdc5d14d543117fb2d8561cb44d8d3362a0309407f7a0a3c0cf7b', 'size': 77931616}, 'starplast-0.55.0-py3-none-any.whl': {'sha256': 'd5741d583e7ffd0784004132961b4133097d56e152c179ddd733369e4f33c198', 'size': 78137077}}, 'installed_wheel': 'verified outside source', 'new_functional_resources': 'exact original bytes', 'broader_goal': 'paused', 'publication': 'pending CI and upload'}